<a href="https://colab.research.google.com/github/AbbhinavJayaraman/HCI-Grad-Course/blob/main/homegrown_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Since ChatGPT and existing LLMs are being annoying, we are just going to follow an existing setup that can be tweaked, making use of ollama since performance on one of our laptops running an modified Fedora Linux distrubtion was able tobe fast and responsive.

Still need to figure out how to connect this with either the PRAAT scripts or myprosody:

Links:
Whisper + ollama + Bark framework: https://medium.com/@vndee.huynh/build-your-own-voice-assistant-and-run-it-locally-whisper-ollama-bark-c80e6f815cba

PRAAT in python: https://github.com/YannickJadoul/Parselmouth
myprosody: https://github.com/Shahabks/myprosody


For all the following packages to install properly, you will need to make sure you on python version 3.12

Install python version 3.12 (we used Homebrew), and then made a python venv with it (python3.12 -m venv sts), where sts is just a name of the venv.

then you can activate the venv using source sts/bin/activate

also make sure you have the portaudio library installed. For Mac, make use of the Homebrew package manager. 

For linux distrubitions, it's best to use the default package manager's version of portaudio, as well as the development headers (for us, that was portaudio-devel on Fedora Silverblue Linux)

In [2]:
# %pip install torch
# %pip install numpy
# %pip install openai-whisper
# %pip install sounddevice
# %pip install rich
# %pip install langchain
# %pip install langchain_community
# %pip install scipy
# %pip install myprosody
# %pip install librosa
# %pip install gtts

In [3]:
import shutil
import os

# Save the STT file locally as generic name
# process transcription
# copy file to archive, or just return and do that after.

# get the output, can just not print also

# Function to delete a file
def delete_file(file_path):
    try:
        os.remove(file_path)
        print(f"File {file_path} deleted successfully.")
    except FileNotFoundError:
        print(f"File {file_path} not found.")
    except PermissionError:
        print(f"Permission denied: {file_path}.")
    except Exception as e:
        print(f"Error occurred while deleting file {file_path}: {e}")


def copy_file (dst: str, src: str) :
    try:
        shutil.copy(src, dst)  # Preserves metadata like timestamps
        print(f"File copied successfully from {src} to {dst}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
import myprosody as mysp
import io
import sys

# Redirect the print output
def detect_sr(src: str) -> int:
    # Create a StringIO object to capture the output
    p=src
    c=r"/home/jayabbhi/Documents/cs395t_hci/lets-talk-tempo/myprosody/myprosody"

    captured_output = io.StringIO()
    sys.stdout = captured_output  # Redirect sys.stdout to the StringIO object
    try:
        # Call the function whose output you want to capture
        mysp.myspsr(p,c)
    finally:
        sys.stdout = sys.__stdout__  # Restore the original sys.stdout

    # Get the captured output as a string
    output = captured_output.getvalue()
    captured_output.close()  # Close the StringIO object
    
    final_syl_sec = 3.0 # a solid defualt value that responses fell into when testing between ourselves, sort of the "i need to talk to Siri but it already messed up twice" rate of speech.
    try:
        final_syl_sec = int(output.split(" ")[1].strip())
    except Exception as e:
        print(e, output)
    
    return final_syl_sec




Before running the program, make sure that ollama is installed (we used Homebrew). You can then use the 'ollama serve' to start Ollama up, use 'ollama pull mistral' to pull a specific LLM (mistral, in this case). 

You do have to specify the name in llm input variable to the ConversationChain.

In [5]:
import whisper
from rich.console import Console
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama

stt = whisper.load_model("tiny")
console = Console()
console = Console(log_path=False)  # Avoid recursive logging


template = """
You are a helpful and friendly AI assistant. You are polite and respectful. You will answer the questions given to you to the best of your ability. You can speak on any topic the user is discussing, no restrictions.
The conversation transcript is as follows:
{history}
And here is the user's follow-up: {input}
Your response:
"""
PROMPT = PromptTemplate(input_variables=["history", "input"], template=template)
chain = ConversationChain(
    prompt=PROMPT,
    verbose=False,
    memory=ConversationBufferMemory(ai_prefix="Assistant:"),
    llm=Ollama(model="llama3.2:1b"),
)

/tmp/ipykernel_15488/1243467133.py:24: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(ai_prefix="Assistant:"),
/tmp/ipykernel_15488/1243467133.py:25: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm=Ollama(model="llama3.2:1b"),
/tmp/ipykernel_15488/1243467133.py:21: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  chain = Conversation

In [6]:
import sounddevice as sd
import librosa
import time
import numpy as np

def record_audio(stop_event, data_queue):
    """
    Captures audio data from the user's microphone and adds it to a queue for further processing.
    Args:
        stop_event (threading.Event): An event that, when set, signals the function to stop recording.
        data_queue (queue.Queue): A queue to which the recorded audio data will be added.
    Returns:
        None
    """
    def callback(indata, frames, time, status):
        if status:
            console.print(status)
        data_queue.put(bytes(indata))

    with sd.RawInputStream(
        samplerate=48000, dtype="int32", channels=1, callback=callback
    ):

        while not stop_event.is_set():
            time.sleep(0.1)

def transcribe(audio_np: np.ndarray) -> str:
    """
    Transcribes the given audio data using the Whisper speech recognition model.
    Args:
        audio_np (numpy.ndarray): The audio data to be transcribed.
    Returns:
        str: The transcribed text.
    """
    result = stt.transcribe(audio_np, fp16=True)  # Set fp16=True if using a GPU
    text = result["text"].strip()
    return text

def get_llm_response(text: str) -> str:
    """
    Generates a response to the given text using the Llama-2 language model.
    Args:
        text (str): The input text to be processed.
    Returns:
        str: The generated response.
    """
    response = chain.predict(input=text)
    if response.startswith("Assistant:"):
        response = response[len("Assistant:") :].strip()
    return response

# def play_audio(audio_file):
#     """
#     Plays the given audio data using the sounddevice library.
#     Args:
#         sample_rate (int): The sample rate of the audio data.
#         audio_array (numpy.ndarray): The audio data to be played.
#     Returns:
#         None
#     """

#     audio = AudioSegment.from_mp3(audio_file)

#     # Convert to numpy array (this gives us access to the raw audio data)
#     audio_data = np.array(audio.get_array_of_samples())

    
#     # Adjust the playback speed by resampling
#     # For example, to double the speed:
#     sd.play(audio_data, audio.frame_rate)
#     sd.wait()

def play_audio(audio_file, user_speech_rate, cur_speech_rate_speedup) -> float: 
    audio_data, sr = librosa.load(audio_file, sr=None)

    # What is our rate of speech in comparison to the default?
    speed_factor = float(user_speech_rate / 2.0) 
    new_speech_rate_speedup = ((speed_factor + cur_speech_rate_speedup) / 2.0)

    # To avoid jarring speed changes, we will average the new speech rate with the old. 

    print(f'New speech rate speedup: {new_speech_rate_speedup}')
    audio_stretched = librosa.effects.time_stretch(audio_data, rate=new_speech_rate_speedup)

    # Play the slowed-down audio
    # sd.play(audio_stretched, sr)
    sd.play(audio_stretched, sr)
    sd.wait()
    return new_speech_rate_speedup

In [8]:
from scipy.io import wavfile
from gtts import gTTS
from queue import Queue
import threading

cur_sr_speedup = 1.0

if __name__ == "__main__":
    console.print("[cyan]Assistant started! Press Ctrl+C to exit.")

    try:
        while True:
            console.input(
                "Press Enter to start recording, then press Enter again to stop."
            )

            data_queue = Queue()  # type: ignore[var-annotated]
            stop_event = threading.Event()
            recording_thread = threading.Thread(
                target=record_audio,
                args=(stop_event, data_queue),
            )
            recording_thread.start()

            input()
            stop_event.set()
            recording_thread.join()

            audio_data = b"".join(list(data_queue.queue))
            audio_np = (np.frombuffer(audio_data, dtype=np.int32))

            if audio_np.size > 0:
                with console.status("Transcribing...", spinner="earth"):
                    # The Audio file exists here, we can pass it along to myprosody
                    wavfile.write('user_en.wav', 48000, audio_np)
                    transcr = (stt.transcribe('user_en.wav'))['text']
                # insert with method to extract the data speech rate method we need
                console.print(f"[yellow]You: {transcr}")

                with console.status("Extracting SR... ", spinner="earth"):
                    copy_file('./myprosody/myprosody/dataset/audioFiles/', 'user_en.wav')
                    user_sr = detect_sr('user_en')
                    # user_sr = 8
                    print(f'User Speech Rate: {user_sr} (syllables per second)')

                with console.status("Generating response...", spinner="earth"):
                    response = get_llm_response(transcr)
                    console.print(f"[cyan]Assistant: {response}")
                    
                # Generate, save, and play the audio
                with console.status("Processing assistant audio..", spinner="earth"):
                    tts = gTTS(text=response, lang='en')
                    tts.save("gtts.mp3")
                    new_speedup = play_audio("gtts.mp3", user_sr, cur_sr_speedup)
                    cur_sr_speedup = new_speedup

                with console.status("Cleaninp up temp files..", spinner="earth"):
                    delete_file("user_en.wav")
                    delete_file("gtts.mp3")
                    delete_file("./myprosody/myprosody/dataset/audioFiles/user_en.wav")
            else:
                console.print(
                    "[red]No audio recorded. Please ensure your microphone is working."
                )

    except KeyboardInterrupt:
        console.print("\n[red]Exiting...")

    console.print("[blue]Session ended.")

Assistant started! Press Ctrl+C to exit.

Press Enter to start recording, then press Enter again to stop.

Output()

You:  Hey, can you tell me about the city of Italy and its people in less than three sentences? I'm considering 
traveling there for a nice senior trip after graduating from my master's degree.

File copied successfully from user_en.wav to ./myprosody/myprosody/dataset/audioFiles/

...

invalid literal for int() with base 10: 'again' Try again the sound of the audio was not clear


Output()


User Speech Rate: 3.0 (syllables per second)


Exiting...

Session ended.